# Translation Quality Evaluation

This notebook runs pairwise evaluation of two Polish translations (A and B) of math problems and solutions.
The LLM is asked to pick the better translation, or declare a tie (`X`).

**Output format per record:** `{ id, problem_verdict, solution_verdict }` — each verdict is `A`, `B`, or `X`.

## 1. Setup

In [1]:
import json
import os
import re
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()

API_KEY  = os.getenv("PCSS_API_KEY", "")
BASE_URL = os.getenv("PCSS_BASE_URL", "https://llm.hpc.psnc.pl/v1/chat/completions")
MODEL    = os.getenv("PCSS_MODEL", "llama3.3:70b")

## 2. LLM helper

In [2]:
def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.0) -> str:
    """Call the OpenAI-compatible API and return the raw text response."""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        "temperature": temperature,
    }
    response = requests.post(BASE_URL, headers=headers, json=payload, timeout=600)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"].strip()

## 3. Evaluation prompts

In [3]:
EVAL_SYSTEM_PROMPT = """\
You are an expert bilingual evaluator specialising in mathematical Polish–English translation quality.
Your sole task is to compare two Polish translations of a mathematical text against the original English,
and decide which translation is superior.

EVALUATION CRITERIA (in order of importance):
1. Mathematical accuracy — all mathematical content, notation, and logic must be preserved exactly.
2. Terminological correctness — standard Polish mathematical terminology must be used
   (e.g. "okrąg opisany" not "okrąg okolony", "symetralna" not "prostopadła dwusieczna").
3. Grammatical correctness — noun cases, adjective agreement, verb forms, and prepositions must be correct
   (e.g. "trójkąta" not "trójkąt" after "długość boku").
4. Naturalness — the Polish should read like text written by a native Polish mathematician,
   not like a literal word-for-word rendering.
5. Completeness — nothing from the original may be omitted or added.

OUTPUT RULES — these are absolute and must never be broken:
- Respond with EXACTLY ONE character: A, B, or X.
- A  → Translation A is better overall.
- B  → Translation B is better overall.
- X  → Both translations are of equal quality (use only when you genuinely cannot distinguish them).
- Do NOT output anything else — no punctuation, no explanation, no newline, nothing.
  A single letter is the complete and correct response.
"""

EVAL_PROBLEM_USER_PROMPT = """\
You will compare two Polish translations of a mathematics problem.
Read the original English, then evaluate both Polish versions.

=== ORIGINAL (English) ===
{problem_en}

=== TRANSLATION A (Polish) ===
{problem_a}

=== TRANSLATION B (Polish) ===
{problem_b}

Which translation is better? Reply with a single letter: A, B, or X (if they are equally good)."""

EVAL_SOLUTION_USER_PROMPT = """\
You will compare two Polish translations of a mathematics solution.
Read the original English, then evaluate both Polish versions.

=== ORIGINAL (English) ===
{solution_en}

=== TRANSLATION A (Polish) ===
{solution_a}

=== TRANSLATION B (Polish) ===
{solution_b}

Which translation is better? Reply with a single letter: A, B, or X (if they are equally good)."""

## 4. Response parser

In [4]:
VALID_VERDICTS = {"A", "B", "X"}

def parse_verdict(raw: str) -> str:
    """
    Extract a single-letter verdict (A / B / X) from the LLM response.
    Strips whitespace and uppercases; raises ValueError if nothing valid is found.
    """
    # Fast path: model obeyed the rules
    candidate = raw.strip().upper()
    if candidate in VALID_VERDICTS:
        return candidate

    # Fallback: find the first occurrence of A, B, or X in the response
    match = re.search(r'\b([ABX])\b', raw.upper())
    if match:
        return match.group(1)

    raise ValueError(f"Could not parse verdict from: {raw!r}")

## 5. I/O helpers

In [5]:
def get_processed_ids(output_file: Path) -> set:
    """Return the set of IDs already written to the output file."""
    if not output_file.exists():
        return set()
    with open(output_file, "r", encoding="utf-8") as f:
        return {json.loads(line)["id"] for line in f if line.strip()}


def save_eval_result(
    output_file: Path,
    idx: int,
    problem_verdict: str,
    solution_verdict: str,
):
    """Append one evaluation result to the JSONL output file."""
    record = {
        "id":               idx,
        "problem_verdict":  problem_verdict,   # A / B / X
        "solution_verdict": solution_verdict,  # A / B / X
    }
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

## 6. Configuration — set your paths here

In [6]:
# ── INPUT FILES ────────────────────────────────────────────────────────────────
# Each file must be a JSONL with records containing:
#   id, problem_en, problem_pl, solution_en, solution_pl
TRANSLATIONS_A_FILE = Path("../data/translations_output.jsonl")       # first run
TRANSLATIONS_B_FILE = Path("../data/translations_output_v2.jsonl")    # second run (different prompts)

# ── OUTPUT FILE ────────────────────────────────────────────────────────────────
# Results are written here as JSONL:
#   { "id": int, "problem_verdict": "A"|"B"|"X", "solution_verdict": "A"|"B"|"X" }
EVAL_OUTPUT_FILE = Path("../data/eval_results.jsonl")

# ── HOW MANY RECORDS TO EVALUATE ──────────────────────────────────────────────
NUM = 110  # set to None to evaluate all records present in both files

print(f"Translation A file : {TRANSLATIONS_A_FILE}")
print(f"Translation B file : {TRANSLATIONS_B_FILE}")
print(f"Output file        : {EVAL_OUTPUT_FILE}")
print(f"Max records        : {NUM}")

Translation A file : ../data/translations_output.jsonl
Translation B file : ../data/translations_output_v2.jsonl
Output file        : ../data/eval_results.jsonl
Max records        : 110


## 7. Load translation datasets

In [7]:
def load_jsonl(path: Path) -> dict:
    """Load a JSONL file and return a dict keyed by record id."""
    records = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                records[rec["id"]] = rec
    return records

translations_a = load_jsonl(TRANSLATIONS_A_FILE)
translations_b = load_jsonl(TRANSLATIONS_B_FILE)

# Only evaluate IDs present in both datasets
common_ids = sorted(translations_a.keys() & translations_b.keys())
if NUM is not None:
    common_ids = common_ids[:NUM]

print(f"Records in A       : {len(translations_a)}")
print(f"Records in B       : {len(translations_b)}")
print(f"Common IDs         : {len(common_ids)}")
print(f"IDs to evaluate    : {len(common_ids)}")

Records in A       : 1251
Records in B       : 110
Common IDs         : 110
IDs to evaluate    : 110


## 8. Run evaluation

In [10]:
%%time

processed = get_processed_ids(EVAL_OUTPUT_FILE)
print(f"Already evaluated  : {len(processed)} records")
print(f"Remaining          : {len(common_ids) - len(processed)}\n")

for idx in common_ids:
    if idx in processed:
        print(f"  Skipping {idx} (already done)")
        continue

    # rec_a = translations_a[idx]
    rec_b = translations_b[idx]

    for i in range(1250):
        rec_a = translations_a[i]
        if rec_a["problem_en"] == rec_b["problem_en"]:
            break
    

    # Sanity-check: both records should share the same original English text
    if rec_a["problem_en"] != rec_b["problem_en"]:
        print(f"  [{idx}] ⚠  problem_en mismatch — skipping")
        continue

    try:
        # ── Evaluate problem translations ──────────────────────────────────────
        raw_problem = call_llm(
            EVAL_SYSTEM_PROMPT,
            EVAL_PROBLEM_USER_PROMPT.format(
                problem_en=rec_a["problem_en"],
                problem_a=rec_a["problem_pl"],
                problem_b=rec_b["problem_pl"],
            ),
        )
        problem_verdict = parse_verdict(raw_problem)

        # ── Evaluate solution translations ─────────────────────────────────────
        raw_solution = call_llm(
            EVAL_SYSTEM_PROMPT,
            EVAL_SOLUTION_USER_PROMPT.format(
                solution_en=rec_a["solution_en"],
                solution_a=rec_a["solution_pl"],
                solution_b=rec_b["solution_pl"],
            ),
        )
        solution_verdict = parse_verdict(raw_solution)

        save_eval_result(EVAL_OUTPUT_FILE, idx, problem_verdict, solution_verdict)
        print(f"  [{idx:>5}] problem={problem_verdict}  solution={solution_verdict}  ✓")

    except Exception as e:
        print(f"  [{idx}] ✗ Error: {e} — skipping")
        continue

print("\nEvaluation complete.")

Already evaluated  : 0 records
Remaining          : 110

  [0] ⚠  problem_en mismatch — skipping
  [1] ⚠  problem_en mismatch — skipping
  [2] ⚠  problem_en mismatch — skipping
  [3] ⚠  problem_en mismatch — skipping
  [4] ⚠  problem_en mismatch — skipping
  [5] ⚠  problem_en mismatch — skipping
  [6] ⚠  problem_en mismatch — skipping
  [7] ⚠  problem_en mismatch — skipping
  [8] ⚠  problem_en mismatch — skipping
  [9] ⚠  problem_en mismatch — skipping
  [10] ⚠  problem_en mismatch — skipping
  [11] ⚠  problem_en mismatch — skipping
  [12] ⚠  problem_en mismatch — skipping
  [13] ⚠  problem_en mismatch — skipping
  [14] ⚠  problem_en mismatch — skipping
  [15] ⚠  problem_en mismatch — skipping
  [16] ⚠  problem_en mismatch — skipping
  [17] ⚠  problem_en mismatch — skipping
  [18] ⚠  problem_en mismatch — skipping
  [19] ⚠  problem_en mismatch — skipping
  [20] ⚠  problem_en mismatch — skipping
  [21] ⚠  problem_en mismatch — skipping
  [22] ⚠  problem_en mismatch — skipping
  [23] ⚠  

## 9. Results summary

In [9]:
from collections import Counter

results = []
with open(EVAL_OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

problem_counts  = Counter(r["problem_verdict"]  for r in results)
solution_counts = Counter(r["solution_verdict"] for r in results)
total = len(results)

print(f"Total evaluated: {total}\n")

print("── Problem translation verdicts ──")
for v in ["A", "B", "X"]:
    n = problem_counts.get(v, 0)
    print(f"  {v}: {n:>5}  ({100*n/total:.1f}%)")

print("\n── Solution translation verdicts ──")
for v in ["A", "B", "X"]:
    n = solution_counts.get(v, 0)
    print(f"  {v}: {n:>5}  ({100*n/total:.1f}%)")

FileNotFoundError: [Errno 2] No such file or directory: '../data/eval_results.jsonl'